# Evaluate best model: Random Forest

In [1]:
# imports
import pickle
import json
import glob
from pathlib import Path
import pandas as pd
import sys

# helpers
sys.path.insert(0, "../../")

from models_helper import ModelTrainer
from dataviz_helper import plot_confusion_matrices

In [2]:
# define directories
DATA_DIR       = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"
MODELS_DIR     = "../../4_modeling/4_5_hyperparameter_tuning/models"
BEST_RUNS_PATH = "../5_1_get_best_runs_wandb/best_by_model_type.json"

# define label columns
LABEL_COLS = [
    "label_ifc_entity",
    "label_predefined_type",
    "label_is_external",
    "label_load_bearing",
]

# load feature names from the best run record
with open(BEST_RUNS_PATH, encoding="utf-8") as f:
    best_runs = json.load(f)

FEATURE_SUBSET = best_runs["random_forest"]["oversampling"]["best_val_f1_macro"]["feature_names"]
print(f"Features ({len(FEATURE_SUBSET)}): {FEATURE_SUBSET}")

Features (37): ['geom_volume', 'geom_surface_area', 'geom_projected_area', 'geom_centroid_x', 'geom_centroid_y', 'geom_centroid_z', 'geom_z_min', 'geom_z_max', 'geom_z_range', 'geom_ratio_area_vol', 'geom_compactness', 'geom_layer_count', 'mat_allgemein', 'mat_aluminium', 'mat_backstein', 'mat_bekleidung', 'mat_belag', 'mat_beton', 'mat_dämm', 'mat_foamglas', 'mat_gips', 'mat_glas', 'mat_holz', 'mat_werkstoff', 'mat_kalksandstein', 'mat_keramik', 'mat_kies', 'mat_kunststein', 'mat_kunststoff', 'mat_metall', 'mat_mörtel', 'mat_naturstein', 'mat_putz', 'mat_stahl', 'mat_zement', 'horizontal_elements_above', 'horizontal_elements_below']


## 1. Load Data and Model

In [3]:
# load validation dataset
df_val = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

X_val = df_val[FEATURE_SUBSET]
y_val = df_val[LABEL_COLS]

print(f"Validation : {len(X_val):,} rows")

Validation : 8,458 rows


In [4]:
# pick the best Random Forest model (highest score in filename and locally stored as pickle)
model_files = glob.glob(f"{MODELS_DIR}/best_rf_model_*.pkl")
model_path  = max(model_files, key=lambda p: float(Path(p).stem.split("_")[-1]))
print(f"Loading: {model_path}")

with open(model_path, "rb") as f:
    model: ModelTrainer = pickle.load(f)

print(f"Model type : {type(model).__name__}")
print(f"Labels     : {model.label_cols}")

Loading: ../../4_modeling/4_5_hyperparameter_tuning/models/best_rf_model_0.7143.pkl
Model type : ModelTrainer
Labels     : ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


## 2. Validation Set Evaluation

In [5]:
# evaluate on validation set to check if everything is working as expected
val_metrics = model.evaluate(X_val, y_val)
val_metrics

,accuracy,precision,recall,mcc,f1_weighted,f1_macro
label,,,,,,
label_ifc_entity,0.8354,0.8333,0.8354,0.7622,0.8027,0.6272
label_predefined_type,0.5073,0.7774,0.5073,0.4797,0.5651,0.4948
label_is_external,0.8461,0.8513,0.8461,0.7543,0.8441,0.8739
label_load_bearing,0.8539,0.8653,0.8539,0.7833,0.8536,0.8614
mean,0.7607,0.8318,0.7607,0.6949,0.7664,0.7143


In [6]:
# compute confusion matrices for all labels on the validation set
val_cms = model.confusion_matrices(X_val, y_val)

plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1)
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1)
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400)

In [7]:
# compute confusion matrices for all labels on the validation set and show them normalized (by true label count)
plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400, normalize=True)